In [ ]:
import importlib
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import stats as st
import plot_helpers as ph

### Read data

In [ ]:
import os

RESULTS_DIR = os.environ.get("RESULTS_DIR", "../results")

gaps_cp_df = st.parse_champsim_output(f"{RESULTS_DIR}/GAP/cp-data")
gaps_wp_df = st.parse_champsim_output(f"{RESULTS_DIR}/GAP/wp-data")
gaps_wpa_df = st.parse_champsim_output(f"{RESULTS_DIR}/GAP/wpa-data")

lcf_cp_df = st.parse_champsim_output(f"{RESULTS_DIR}/LCF/cp-data")
lcf_wp_df = st.parse_champsim_output(f"{RESULTS_DIR}/LCF/wp-data")
lcf_wpa_df = st.parse_champsim_output(f"{RESULTS_DIR}/LCF/wpa-data")

spec_cp_df = st.merge_simpoints(st.parse_champsim_output(f"{RESULTS_DIR}/SPEC/cp-data"))
spec_wp_df = st.merge_simpoints(st.parse_champsim_output(f"{RESULTS_DIR}/SPEC/wp-data"))
spec_wpa_df = st.merge_simpoints(
    st.parse_champsim_output(f"{RESULTS_DIR}/SPEC/wpa-data")
)

### Group data

In [ ]:
groupings = {
    # Default config
    "default": "default",
    # L1I Prefetchers
    "l1i-next_line": "l1i-next_line",
    "l1i-barsa": "l1i-barsa",
    "l1i-bip": "l1i-bip",
    "l1i-djolt": "l1i-djolt",
    "l1i-epi": "l1i-epi",
    "l1i-fnlmma": "l1i-fnlmma_new",
    "l1i-mana": "l1i-mana",
    "l1i-pips": "l1i-pips",
    "l1i-tap": "l1i-tap",
    # L1D Prefetchers
    "l1d-next_line": "l1d-next_line",
    "l1d-ip_stride": "l1d-ip_stride",
    "l1d-berti": "l1d-berti",
    "l1d-ipcp": "l1d-ipcp",
    # L2C Prefetchers
    "l2c-next_line": "l2c-next_line",
    "l2c-ip_stride": "l2c-ip_stride",
    "l2c-spp": "l2c-spp_dev",
    "l2c-ampm": "l2c-ampm",
    "l2c-bingo": "l2c-bingo",
    "l2c-bop": "l2c-bop",
    "l2c-dspatch": "l2c-dspatch",
    "l2c-ipcp": "l2c-ipcp",
    "l2c-mlop": "l2c-mlop",
    "l2c-sandbox": "l2c-sandbox",
    "l2c-scooby": "l2c-scooby",
    "l2c-sms": "l2c-sms",
    "l2c-spp_ppf": "l2c-spp_ppf_dev",
    "l2c-streamer": "l2c-streamer",
    # LLC Prefetchers
    "llc-next_line": "llc-next_line",
    "llc-ip_stride": "llc-ip_stride",
    # LLC Replacement Policies
    "llc-ship": "ship",
    "llc-srrip": "srrip",
    "llc-drrip": "drrip",
    "llc-mockingjay": "mockingjay",
}

datasets = {
    "gaps": {"cp": gaps_cp_df, "wp": gaps_wp_df, "wpa": gaps_wpa_df},
    "lcf": {"cp": lcf_cp_df, "wp": lcf_wp_df, "wpa": lcf_wpa_df},
    "spec": {"cp": spec_cp_df, "wp": spec_wp_df, "wpa": spec_wpa_df},
}

results = {}
for category, versions in datasets.items():
    for version, df in versions.items():
        # Extract algorithm suffix once per DataFrame
        algos = df.index.str.replace(r"^.*?champsim-", "", regex=True)
        for label, group_by_str in groupings.items():
            mask = algos == group_by_str
            if not mask.any():
                continue
            grouped = df[mask].copy()
            grouped.index = grouped.index.str.replace(
                f"-champsim-{group_by_str}", "", regex=False
            )
            grouped = grouped.sort_index()
            key = f"{category}_{version}_{label}_df"
            results[key] = st.calculate_means(grouped)

### L1I Instruction Footprint

In [ ]:
importlib.reload(ph)

# -- Config --
COL = "instr_foot_print"
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
WP_AMEAN_HATCH = "/"
WPA_AMEAN_HATCH = "\\"
Y_MIN = 0
Y_MAX = 180
Y_DTICK = 60
FIG_NAME = "instr_footprint"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def footprint_pct_increase(version):
    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][COL]
        other = results[f"{suite}_{version}_default_df"][COL]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val - cp_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, wp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    footprint_pct_increase("wp"),
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    footprint_pct_increase("wpa"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# Amean hatching
wp_patterns = [""] * len(display_names)
wpa_patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "amean" in name:
        wp_patterns[i] = WP_AMEAN_HATCH
        wpa_patterns[i] = WPA_AMEAN_HATCH

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wpa_values,
        marker_color=WPA_BAR_COLOR,
        name="TI-WP",
    )
)

fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(dtick=Y_DTICK, range=[Y_MIN, Y_MAX]),
        legend=dict(
            orientation="v",
            x=0.015,
            y=0.95,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1I Data Footprint

In [ ]:
importlib.reload(ph)

# -- Config --
COL = "data_foot_print"
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
WP_AMEAN_HATCH = "/"
WPA_AMEAN_HATCH = "\\"
Y_MIN = 0
Y_MAX = 14.1
Y_DTICK = 2
FIG_NAME = "data_footprint"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def footprint_pct_increase(version):
    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][COL]
        other = results[f"{suite}_{version}_default_df"][COL]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val - cp_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, wp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    footprint_pct_increase("wp"),
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    footprint_pct_increase("wpa"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# Amean hatching
wp_patterns = [""] * len(display_names)
wpa_patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "amean" in name:
        wp_patterns[i] = WP_AMEAN_HATCH
        wpa_patterns[i] = WPA_AMEAN_HATCH

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wpa_values,
        marker_color=WPA_BAR_COLOR,
        name="TI-WP",
    )
)

fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(dtick=Y_DTICK, range=[Y_MIN, Y_MAX]),
        legend=dict(
            orientation="v",
            x=0.015,
            y=0.95,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### Wrong-Path Data Fills

In [26]:
importlib.reload(ph)

# -- Config --
LEVELS = ["L1D"]
LEVEL_COLORS = {
    "L1D": "darkorange",
}
Y_MIN = 0
Y_MAX = None
FIG_NAME = "wp_data_fills"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def wp_fills_pct(level, version):
    fill_col = f"{level}_WRONG_PATH_FILL"
    miss_col = f"{level}_POLLUTION_CP_MISS"

    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"]
        other = results[f"{suite}_{version}_default_df"]
        vals = []
        for bm in benchmarks:
            fills = other[fill_col].get(bm, 0)
            cp_miss = cp[miss_col].get(bm, 0)
            vals.append((fills / cp_miss * 100) if cp_miss != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, _, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    wp_fills_pct(LEVELS[0], "wp"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

wp_vals = {}
wpa_vals = {}
for level in LEVELS:
    _, wp_vals[level], _, _ = ph.get_metric_across_suites(
        results,
        wp_fills_pct(level, "wp"),
    )
    _, wpa_vals[level], _, _ = ph.get_metric_across_suites(
        results,
        wp_fills_pct(level, "wpa"),
    )

# -- Plot --
LEVEL_PATTERNS = {"L1D": ""}

fig = go.Figure()
for level in LEVELS:
    pat = LEVEL_PATTERNS[level]
    col = LEVEL_COLORS[level]
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wpa_vals[level],
            marker_color="steelblue",
            name="TI-WP",
        )
    )
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wp_vals[level],
            marker=dict(
                color="rgba(0,0,0,0)",
                line=dict(color="darkorange", width=2),
                pattern=dict(shape="/", fgcolor="darkorange", solidity=0.4),
            ),
            name="ED-WP",
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(),
        bargap=0,
        bargroupgap=0,
        legend=dict(
            orientation="v",
            x=0.015,
            y=0.95,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### Instruction Overhead & MPKI

In [ ]:
importlib.reload(ph)

# -- Config --
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
Y_MIN = 0
Y_MAX = 250
Y_DTICK = 50
FIG_NAME = "overhead_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE
SIM_INSTS = 100_000_000
MPKI_COLORS = {"lcf": "black", "gaps": "black", "spec": "black"}


# -- Metric --
def extra_instr_pct(version):
    def fn(results, suite, benchmarks):
        wp = results[f"{suite}_{version}_default_df"]["wrong_path_insts_executed"]
        return [(wp.get(bm, 0) / SIM_INSTS * 100) for bm in benchmarks]

    return fn


def mpki_values_fn(results, suite, benchmarks):
    mpki = results[f"{suite}_wp_default_df"]["MPKI"]
    return [mpki.get(bm, 0) for bm in benchmarks]


# -- Data --
all_benchmarks, wp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    extra_instr_pct("wp"),
    include_amean=False,
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    extra_instr_pct("wpa"),
    include_amean=False,
)
_, mpki_values, _, _ = ph.get_metric_across_suites(
    results,
    mpki_values_fn,
    include_amean=False,
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# -- Suite boundary indices for MPKI lines --
bounds = [0] + [int(b + 0.5) for b in suite_boundaries]

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=list(range(len(display_names))),
        y=wp_values,
        marker_color=WP_BAR_COLOR,
        name="WP",
    )
)
# fig.add_trace(
#     go.Bar(
#         x=display_names,
#         y=wpa_values,
#         marker_color=WPA_BAR_COLOR,
#         name="WPA",
#     )
# )

for i, suite in enumerate(ph.SUITES):
    s, e = bounds[i], bounds[i + 1]
    fig.add_trace(
        go.Scatter(
            x=list(range(s, e)),
            y=mpki_values[s:e],
            mode="lines+markers",
            name=f"MPKI {ph.SUITE_DISPLAY[suite]}",
            showlegend=False,
            line=dict(color=MPKI_COLORS[suite]),
            yaxis="y2",
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(
            tickvals=list(range(len(display_names))),
            ticktext=tick_labels,
            range=[-0.5, len(display_names) - 0.5],
        ),
        showlegend=False,
        yaxis=dict(
            title="Extra Instr. (%)",
            range=[Y_MIN, Y_MAX],
            tickvals=[0, 50, 100, 150, 200],
        ),
        yaxis2=dict(
            title="Branch MPKI",
            overlaying="y",
            side="right",
            showgrid=True,
            gridcolor="lightgray",
            gridwidth=0.5,
            zeroline=False,
            griddash="dash",
            range=[0, 40],
            tickvals=[0, 8, 16, 24, 32],
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1I Hits & Misses

In [ ]:
importlib.reload(ph)

# -- Config --
HITS_COLOR = "mediumseagreen"
MISSES_COLOR = "salmon"
AMEAN_HATCH = "/"
Y_MIN = 0
Y_MAX = 450.1
Y_DTICK = 100
FIG_NAME = "l1i_hits_misses_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def cache_pct_increase(col, version):
    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][col]
        other = results[f"{suite}_{version}_default_df"][col]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val - cp_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, wp_hits, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    cache_pct_increase("L1I_TOTAL_HITS", "wp"),
)
_, wp_misses, _, _ = ph.get_metric_across_suites(
    results,
    cache_pct_increase("L1I_TOTAL_MISS", "wp"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_hits,
        marker_color=HITS_COLOR,
        name="Hits",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_misses,
        marker_color=MISSES_COLOR,
        name="Misses",
    )
)

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

# Overflow annotations — hits with arrows, misses centered no arrows
_hits_ax = [-50, -30, 0, 30, 50]
_hits_xshift = [-10, -10, -10, -10, -10]
_hits_xanchor = ["right", "center", "center", "center", "left"]
_hits_idx = 0
for i, val in enumerate(wp_hits):
    if val > Y_MAX:
        fig.add_annotation(
            x=display_names[i],
            y=Y_MAX - 10,
            text=f"{val:.0f}",
            showarrow=True,
            arrowhead=2,
            arrowwidth=2,
            xshift=_hits_xshift[_hits_idx] if _hits_idx < len(_hits_xshift) else 0,
            ax=_hits_ax[_hits_idx] if _hits_idx < len(_hits_ax) else 0,
            ay=-30,
            xanchor=(
                _hits_xanchor[_hits_idx] if _hits_idx < len(_hits_xanchor) else "center"
            ),
            font=dict(size=FONT_SIZE * 0.85),
            textangle=0,
        )
        _hits_idx += 1

for i, val in enumerate(wp_misses):
    if val > Y_MAX:
        fig.add_annotation(
            x=display_names[i],
            y=Y_MAX - 5,
            text=f"{val:.0f}",
            showarrow=False,
            font=dict(size=FONT_SIZE * 0.85),
            yshift=20,
            xshift=7,
            textangle=0,
        )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(range=[Y_MIN, Y_MAX], dtick=Y_DTICK),
        legend=dict(
            orientation="v",
            x=0.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1D, L2C, LLC Misses

In [ ]:
importlib.reload(ph)

# -- Config --
LEVELS = ["L1D", "L2C"]
LEVEL_COLORS = {"L1D": "orange", "L2C": "dodgerblue"}
Y_MIN = 0
Y_MAX = 60
FIG_NAME = "misses_L1D_L2C"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def misses_pct_increase(level, version):
    col = f"{level}_TOTAL_MISS"

    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][col]
        other = results[f"{suite}_{version}_default_df"][col]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val - cp_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, _, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    misses_pct_increase(LEVELS[0], "wp"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

wp_misses = {}
wpa_misses = {}
for level in LEVELS:
    _, wp_misses[level], _, _ = ph.get_metric_across_suites(
        results,
        misses_pct_increase(level, "wp"),
    )
    _, wpa_misses[level], _, _ = ph.get_metric_across_suites(
        results,
        misses_pct_increase(level, "wpa"),
    )

# -- Plot --
fig = go.Figure()
for level in LEVELS:
    col = LEVEL_COLORS[level]
    # TI-WP: solid colored
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wpa_misses[level],
            marker=dict(
                color=col,
                line=dict(color="black", width=0.5),
            ),
            name=f"TI-WP {level}",
        )
    )
    # ED-WP: hollow with colored hatch
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wp_misses[level],
            marker=dict(
                color="rgba(0,0,0,0)",
                line=dict(color=col, width=2),
                pattern=dict(shape="/", fgcolor=col, solidity=0.4),
            ),
            name=f"ED-WP {level}",
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

# Overflow annotations (verilator) — no tilt
for level in LEVELS:
    for values in [wp_misses[level], wpa_misses[level]]:
        for i, val in enumerate(values):
            if val > Y_MAX:
                fig.add_annotation(
                    x=display_names[i],
                    y=Y_MAX - 2,
                    text=f"{val:.0f}",
                    showarrow=False,
                    font=dict(size=FONT_SIZE * 0.85),
                    yshift=10,
                    textangle=0,
                )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(range=[Y_MIN, Y_MAX]),
        legend=dict(
            orientation="h",
            x=0.5,
            y=1.01,
            xanchor="center",
            yanchor="bottom",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1D, L2C, LLC Hits

In [ ]:
importlib.reload(ph)

# -- Config --
LEVELS = ["L1D", "L2C"]
LEVEL_COLORS = {"L1D": "orange", "L2C": "dodgerblue"}
Y_MIN = 0
Y_MAX = 201.1
FIG_NAME = "hits_L1D_L2C"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def hits_pct_increase(level, version):
    col = f"{level}_TOTAL_HITS"

    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][col]
        other = results[f"{suite}_{version}_default_df"][col]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val - cp_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, _, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    hits_pct_increase(LEVELS[0], "wp"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

wp_hits = {}
wpa_hits = {}
for level in LEVELS:
    _, wp_hits[level], _, _ = ph.get_metric_across_suites(
        results,
        hits_pct_increase(level, "wp"),
    )
    _, wpa_hits[level], _, _ = ph.get_metric_across_suites(
        results,
        hits_pct_increase(level, "wpa"),
    )

# -- Plot --
fig = go.Figure()
for level in LEVELS:
    col = LEVEL_COLORS[level]
    # TI-WP: solid colored
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wpa_hits[level],
            marker=dict(
                color=col,
                line=dict(color="black", width=0.5),
            ),
            name=f"TI-WP {level}",
        )
    )
    # ED-WP: hollow with colored hatch
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wp_hits[level],
            marker=dict(
                color="rgba(0,0,0,0)",
                line=dict(color=col, width=2),
                pattern=dict(shape="/", fgcolor=col, solidity=0.4),
            ),
            name=f"ED-WP {level}",
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

# Overflow annotations — TI-WP L1D (ccsv): centered, no arrow
for i, val in enumerate(wpa_hits["L1D"]):
    if val > Y_MAX:
        fig.add_annotation(
            x=display_names[i],
            y=Y_MAX - 10,
            xshift=-15,
            text=f"{val:.0f}",
            showarrow=True,
            arrowhead=2,
            arrowwidth=2,
            ax=-50,
            ay=-15,
            xanchor="right",
            font=dict(size=FONT_SIZE * 0.85),
            yshift=10,
            textangle=0,
        )

# TI-WP L2C (verilator)
for i, val in enumerate(wpa_hits["L2C"]):
    if val > Y_MAX:
        fig.add_annotation(
            x=display_names[i],
            y=Y_MAX,
            text=f"{val:.0f}",
            showarrow=True,
            arrowhead=2,
            arrowwidth=2,
            ax=-30,
            ay=10,
            xanchor="right",
            font=dict(size=FONT_SIZE * 0.85),
            textangle=0,
        )

# ED-WP L2C (verilator)
for i, val in enumerate(wp_hits["L2C"]):
    if val > Y_MAX:
        fig.add_annotation(
            x=display_names[i],
            y=Y_MAX,
            text=f"{val:.0f}",
            showarrow=True,
            arrowhead=2,
            arrowwidth=2,
            ax=30,
            ay=10,
            xshift=25,
            xanchor="left",
            font=dict(size=FONT_SIZE * 0.85),
            textangle=0,
        )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(range=[Y_MIN, Y_MAX]),
        legend=dict(
            orientation="h",
            x=0.5,
            y=1.10,
            xanchor="center",
            yanchor="bottom",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### ROB at Misspredict

In [ ]:
importlib.reload(ph)

# -- Config --
CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
AMEAN_HATCH = "/"
FIG_NAME = "rob_plot"
Y_DTICK = 100
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def rob_at_mispredict(version):
    def fn(results, suite, benchmarks):
        rob = results[f"{suite}_{version}_default_df"][
            "Average_ROB_Occupancy_at_Mispredict"
        ]
        return [rob.get(bm, 0) for bm in benchmarks]

    return fn


# -- Data --
all_benchmarks, cp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    rob_at_mispredict("cp"),
)
_, wp_values, _, _ = ph.get_metric_across_suites(
    results,
    rob_at_mispredict("wp"),
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    rob_at_mispredict("wpa"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# Amean hatching
patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "amean" in name:
        patterns[i] = AMEAN_HATCH

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=cp_values,
        marker_color=CP_BAR_COLOR,
        name="No-WP",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)
# fig.add_trace(
#     go.Bar(
#         x=display_names,
#         y=wpa_values,
#         marker_color=WPA_BAR_COLOR,
#         marker_pattern_shape=patterns,
#         name="WPA",
#     )
# )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(dtick=100),
        legend=dict(
            orientation="v",
            x=0.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### ROB Full Events

In [ ]:
importlib.reload(ph)

# -- Config --
CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
AMEAN_HATCH = "/"
Y_MAX = 25
FIG_NAME = "rob_pki"
FONT_SIZE = ph.ISPASS_FONT_SIZE
SIM_INSTS_K = 100_000  # 100M instructions / 1000


# -- Metric --
def rob_full_pki(version):
    def fn(results, suite, benchmarks):
        rob_full = results[f"{suite}_{version}_default_df"]["ROB_Full_Events"]
        return [rob_full.get(bm, 0) / SIM_INSTS_K for bm in benchmarks]

    return fn


# -- Data --
all_benchmarks, cp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    rob_full_pki("cp"),
)
_, wp_values, _, _ = ph.get_metric_across_suites(
    results,
    rob_full_pki("wp"),
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    rob_full_pki("wpa"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# Amean hatching
patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "amean" in name:
        patterns[i] = AMEAN_HATCH

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=cp_values,
        marker_color=CP_BAR_COLOR,
        name="No-WP",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)
# fig.add_trace(
#     go.Bar(
#         x=display_names,
#         y=wpa_values,
#         marker_color=WPA_BAR_COLOR,
#         marker_pattern_shape=patterns,
#         name="WPA",
#     )
# )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

# Overflow annotations
_ann_ax = [-30, -10, 10, 30]
_ann_xshift = [-15, -5, 5, 15]
_ann_xanchor = ["right", "left", "right", "left"]
_ann_idx = 0
for values in [cp_values, wp_values]:
    for i, val in enumerate(values):
        if val > Y_MAX:
            fig.add_annotation(
                x=display_names[i],
                xshift=_ann_xshift[_ann_idx] if _ann_idx < len(_ann_xshift) else 0,
                y=Y_MAX - 0.5,
                text=f"{val:.0f}",
                showarrow=True,
                arrowhead=2,
                arrowwidth=2,
                ax=_ann_ax[_ann_idx] if _ann_idx < len(_ann_ax) else 0,
                ay=-30,
                xanchor=(
                    _ann_xanchor[_ann_idx] if _ann_idx < len(_ann_xanchor) else "center"
                ),
                font=dict(size=FONT_SIZE * 0.85),
                textangle=0,
            )
            _ann_idx += 1

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        margin=dict(t=60),
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(range=[0, Y_MAX]),
        legend=dict(
            orientation="v",
            x=0.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### Execute-Stage Activity Breakdown

In [ ]:
importlib.reload(ph)

# -- Config --
FIG_NAME = "stacked_cycles"
FONT_SIZE = ph.ISPASS_FONT_SIZE

CATEGORIES = {
    "CP": {"col": "Execute_Only_CP_Cycles", "color": "mediumseagreen", "pattern": ""},
    "WP": {"col": "Execute_Only_WP_Cycles", "color": "darkorange", "pattern": "/"},
    "CP+WP": {"col": "Execute_CP_WP_Cycles", "color": "steelblue", "pattern": ""},
    "ROB Empty": {
        "col": ["Execute_ROB_Empty_Cycles", "Execute_ROB_Empty_Repair_Cycles"],
        "color": "white",
        "pattern": "\\",
    },
    "ROB Empty (No WP)": {
        "col": "Execute_ROB_Empty_Fetch_Stalled_No_WP_Cycles",
        "color": "plum",
        "pattern": "",
    },
    "ROB Not Ready": {
        "col": [
            "Execute_ROB_Not_Ready_Not_Full_New_Added_Cycles",
            "Execute_ROB_Not_Ready_Not_Full_No_New_Added_Cycles",
            "Execute_ROB_Not_Ready_Full_New_Added_Cycles",
            "Execute_ROB_Not_Ready_Full_No_New_Added_Cycles",
        ],
        "color": "gold",
        "pattern": "x",
    },
}


# -- Metric --
def cycle_fraction(category):
    cols = CATEGORIES[category]["col"]
    if isinstance(cols, str):
        cols = [cols]

    def fn(results, suite, benchmarks):
        total = results[f"{suite}_wp_default_df"]["total_cycles"]
        vals = []
        for bm in benchmarks:
            numerator = sum(
                results[f"{suite}_wp_default_df"][c].get(bm, 0) for c in cols
            )
            t = total.get(bm, 1)
            vals.append(numerator / t if t != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, _, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    cycle_fraction("CP"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

cat_data = {}
for cat in CATEGORIES:
    _, cat_data[cat], _, _ = ph.get_metric_across_suites(
        results,
        cycle_fraction(cat),
    )

# -- Plot --
fig = go.Figure()
for cat, props in CATEGORIES.items():
    pat = props.get("pattern", "")
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=cat_data[cat],
            name=cat,
            marker=dict(
                color=props["color"],
                line=dict(color="black", width=0.5),
                pattern=(
                    dict(shape=pat, fgcolor="black", solidity=0.3) if pat else dict()
                ),
            ),
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)
fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        barmode="stack",
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(title="Proportion of Total Cycles", tickformat=".0%"),
        legend=dict(
            y=0.97,
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_TWO_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### Speedup for all benchmarks

In [ ]:
importlib.reload(ph)

# -- Config --
WP_BAR_COLOR = "darkorange"
WPA_BAR_COLOR = "steelblue"
AMEAN_HATCH = "/"

Y_MAX = 25
FIG_NAME = "speedup_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def ipc_speedup(version):
    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"]["IPC"]
        other = results[f"{suite}_{version}_default_df"]["IPC"]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((other_val / cp_val) - 1) * 100 if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, wp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    ipc_speedup("wp"),
    include_amean=False,
    include_gmean=True,
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    ipc_speedup("wpa"),
    include_amean=False,
    include_gmean=True,
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# Gmean hatching
wp_patterns = [""] * len(display_names)
wpa_patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "gmean" in name:
        wp_patterns[i] = "/"
        wpa_patterns[i] = "\\"

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wpa_values,
        marker_color=WPA_BAR_COLOR,
        name="TI-WP",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

# Overflow annotations: 1st,2nd shift left; 3rd,4th shift right
_ann_ax = [-30, -15, 15, 30]
_ann_xanchor = ["right", "center", "center", "left"]
_ann_idx = 0
for values in [wp_values]:
    for i, val in enumerate(values):
        if val > Y_MAX:
            fig.add_annotation(
                x=display_names[i],
                y=Y_MAX - 0.5,
                text=f"{val:.0f}",
                showarrow=True,
                arrowhead=2,
                arrowwidth=2,
                ax=_ann_ax[_ann_idx] if _ann_idx < len(_ann_ax) else 0,
                ay=-30,
                xanchor=(
                    _ann_xanchor[_ann_idx] if _ann_idx < len(_ann_xanchor) else "center"
                ),
                font=dict(size=30),
                textangle=0,
            )
            _ann_idx += 1

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        margin=dict(t=60),
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(
            title="Speedup (%)", range=[-5, Y_MAX], tickvals=[0, 5, 10, 15, 20, 25]
        ),
        legend=dict(
            orientation="v",
            x=0.02,
            y=1.1,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1I, L1D, L2C, LLC CP

In [ ]:
importlib.reload(ph)

# -- Config --
LEVELS = ["L1I", "L1D", "L2C"]
LEVEL_COLORS = {
    "L1I": "mediumseagreen",
    "L1D": "darkorange",
    "L2C": "steelblue",
}
AMEAN_HATCH = "/"
Y_MIN = -20
Y_MAX = 100
FIG_NAME = "L1I_L1D_L2C"
FONT_SIZE = ph.ISPASS_FONT_SIZE


# -- Metric --
def pollution_cp_miss_reduction(level, version):
    col = f"{level}_POLLUTION_CP_MISS"

    def fn(results, suite, benchmarks):
        cp = results[f"{suite}_cp_default_df"][col]
        other = results[f"{suite}_{version}_default_df"][col]
        vals = []
        for bm in benchmarks:
            cp_val = cp.get(bm, 0)
            other_val = other.get(bm, 0)
            vals.append(((cp_val - other_val) / cp_val * 100) if cp_val != 0 else 0)
        return vals

    return fn


# -- Data --
all_benchmarks, _, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    pollution_cp_miss_reduction(LEVELS[0], "wp"),
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

wp_vals = {}
wpa_vals = {}
for level in LEVELS:
    _, wp_vals[level], _, _ = ph.get_metric_across_suites(
        results,
        pollution_cp_miss_reduction(level, "wp"),
    )
    _, wpa_vals[level], _, _ = ph.get_metric_across_suites(
        results,
        pollution_cp_miss_reduction(level, "wpa"),
    )

# Amean hatching
patterns = [""] * len(display_names)
for i, name in enumerate(display_names):
    if "amean" in name:
        patterns[i] = AMEAN_HATCH

# -- Plot --
# TI-WP: black filled, with white hatch per level
# ED-WP: white filled, with black hatch per level (same patterns)
LEVEL_PATTERNS = {"L1I": "", "L1D": "\\", "L2C": "-", "LLC": "-"}

fig = go.Figure()
for level in LEVELS:
    pat = LEVEL_PATTERNS[level]
    col = LEVEL_COLORS[level]
    # TI-WP: solid gray
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wpa_vals[level],
            marker=dict(
                color=col,
                line=dict(color="black", width=0.5),
            ),
            name=f"TI-WP {level}",
        )
    )
    # ED-WP: hollow with black outline and hatch
    fig.add_trace(
        go.Bar(
            x=display_names,
            y=wp_vals[level],
            marker=dict(
                color="rgba(0,0,0,0)",
                line=dict(color=col, width=2),
                pattern=dict(shape=pat if pat else "/", fgcolor=col, solidity=0.4),
            ),
            name=f"ED-WP {level}",
        )
    )

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(title="CP Miss Reduction (%)", range=[Y_MIN, Y_MAX]),
        bargap=0,
        bargroupgap=0,
        legend=dict(
            y=0.97,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_TWO_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1I CP v WP

In [ ]:
import math

importlib.reload(ph)

# -- Config --
FIG_NAME = "l1i_prefetchers_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE
SUITE = "lcf"
CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"

PREFETCHERS = {
    "l1i-barsa": "BARÇA",
    "l1i-bip": "BIP",
    "l1i-djolt": "Djolt",
    "l1i-epi": "EIP",
    "l1i-fnlmma": "FNL+MMA",
    "l1i-mana": "Mana",
    "l1i-next_line": "Next Line",
    "l1i-pips": "PIPS",
    "l1i-tap": "TAP",
}


# -- Compute speedups --
def get_pf_speedups(version):
    baseline = results[f"{SUITE}_{version}_default_df"]["IPC"]["gmean"]
    labels, speedups = [], []
    for pf, label in PREFETCHERS.items():
        pf_ipc = (
            results.get(f"{SUITE}_{version}_{pf}_df", {})
            .get("IPC", {})
            .get("gmean", None)
        )
        if pf_ipc is None:
            continue
        labels.append(label)
        speedups.append(((pf_ipc / baseline) - 1) * 100)
    return labels, speedups


cp_labels, cp_speedups = get_pf_speedups("cp")
wp_labels, wp_speedups = get_pf_speedups("wp")

all_speedups = cp_speedups + wp_speedups
Y_MIN = -8.0
Y_MAX = 10.1
Y_DTICK = 3

# -- Plot --
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=cp_labels,
        y=cp_speedups,
        marker_color=CP_BAR_COLOR,
        name="No-WP",
    )
)

fig.add_trace(
    go.Bar(
        x=wp_labels,
        y=wp_speedups,
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
        name="ED-WP",
    )
)

# Suite label
fig.add_annotation(
    x=0.5,
    y=1,
    text="LCF",
    showarrow=False,
    xref="paper",
    yref="paper",
    font=dict(size=FONT_SIZE, color="black"),
)

# fig.add_annotation(
#     x=0.99,
#     y=1.05,
#     text="(a)",
#     showarrow=False,
#     xref="paper",
#     yref="paper",
#     font=dict(size=FONT_SIZE, color="black", family="Arial, sans-serif"),
#     xanchor="right",
#     yanchor="top",
# )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        margin=dict(b=80),
        xaxis=dict(tickangle=-25),
        yaxis=dict(title="Speedup (%)", range=[Y_MIN, Y_MAX], dtick=Y_DTICK, tick0=0),
        legend=dict(
            orientation="v",
            x=0.20,
            y=1.0,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L1D CP v WP

In [ ]:
import math

importlib.reload(ph)

# -- Config --
FIG_NAME = "l1d_prefetchers_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE

PREFETCHERS = {
    "l1d-berti": "Berti",
    "l1d-ip_stride": "IP Stride",
    "l1d-ipcp": "IPCP",
    "l1d-next_line": "Next Line",
}

CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"


# -- Compute speedups --
def get_pf_speedups(version):
    speedups = {}
    for pf, label in PREFETCHERS.items():
        vals = {}
        for suite in ph.SUITES:
            baseline = results[f"{suite}_{version}_default_df"]["IPC"]["gmean"]
            pf_ipc = results[f"{suite}_{version}_{pf}_df"]["IPC"]["gmean"]
            vals[suite] = ((pf_ipc / baseline) - 1) * 100
        speedups[pf] = vals
    return speedups


cp_data = get_pf_speedups("cp")
wp_data = get_pf_speedups("wp")

# -- Build x-axis: prefetcher names repeated per suite --
all_names = []  # display names for x
all_cp_vals = []
all_wp_vals = []
suite_boundaries = []
suite_labels = []
x_pos = 0

for suite in ph.SUITES:
    start = x_pos
    for pf, label in PREFETCHERS.items():
        all_names.append(label)
        all_cp_vals.append(cp_data[pf][suite])
        all_wp_vals.append(wp_data[pf][suite])
        x_pos += 1
    suite_labels.append((start + len(PREFETCHERS) / 2, ph.SUITE_DISPLAY[suite]))
    suite_boundaries.append(x_pos - 0.5)

all_vals = all_cp_vals + all_wp_vals
Y_MIN = -3.1
Y_MAX = 9.1
Y_DTICK = 3

# -- Plot --
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_cp_vals,
        name="No-WP",
        marker_color=CP_BAR_COLOR,
    )
)

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_wp_vals,
        name="ED-WP",
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
    )
)

# Suite dividers and labels
for boundary in suite_boundaries[:-1]:
    fig.add_vline(x=boundary, line_width=2, line_dash="dash", line_color="gray")
for xpos, label in suite_labels:
    fig.add_annotation(
        x=xpos,
        y=0.95,
        text=label,
        showarrow=False,
        xref="x",
        yref="paper",
        font=dict(size=FONT_SIZE, color="black"),
    )


# Overflow annotations
_ann_ax = [-50, 50]
_ann_xshift = [-15, 15]
_ann_xanchor = ["right", "left"]
_ann_idx = 0
for values in [all_cp_vals, all_wp_vals]:
    for i, val in enumerate(values):
        if val > Y_MAX or val < Y_MIN:
            fig.add_annotation(
                x=i,
                xshift=_ann_xshift[_ann_idx] if _ann_idx < len(_ann_xshift) else 0,
                y=-3,
                text=f"{val:.1f}",
                showarrow=True,
                arrowhead=2,
                arrowwidth=2,
                ax=_ann_ax[_ann_idx] if _ann_idx < len(_ann_ax) else 0,
                ay=-30 if val > 0 else -30,
                xanchor=(
                    _ann_xanchor[_ann_idx] if _ann_idx < len(_ann_xanchor) else "center"
                ),
                font=dict(size=FONT_SIZE * 0.85),
                textangle=0,
            )
            _ann_idx += 1

# fig.add_annotation(
#     x=0.99,
#     y=1.05,
#     text="(b)",
#     showarrow=False,
#     xref="paper",
#     yref="paper",
#     font=dict(size=FONT_SIZE, color="black", family="Arial, sans-serif"),
#     xanchor="right",
#     yanchor="top",
# )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(
            tickvals=list(range(len(all_names))),
            ticktext=all_names,
            range=[-0.5, len(all_names) - 0.5],
        ),
        yaxis=dict(title="Speedup (%)", range=[Y_MIN, Y_MAX], dtick=Y_DTICK, tick0=0),
        showlegend=False,
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### L2C CP v WP

In [ ]:
import math

importlib.reload(ph)

# -- Config --
FIG_NAME = "l2c_prefetchers_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE

PREFETCHERS = {
    "l2c-bingo": "Bingo",
    "l2c-ip_stride": "IP Stride",
    "l2c-mlop": "MLOP",
    "l2c-next_line": "Next Line",
    "l2c-sms": "SMS",
    "l2c-spp": "SPP",
    "l2c-streamer": "Streamer",
}

CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"


# -- Compute speedups --
def get_pf_speedups(version):
    speedups = {}
    for pf, label in PREFETCHERS.items():
        vals = {}
        for suite in ph.SUITES:
            baseline = results[f"{suite}_{version}_default_df"]["IPC"]["gmean"]
            pf_ipc = results[f"{suite}_{version}_{pf}_df"]["IPC"]["gmean"]
            vals[suite] = ((pf_ipc / baseline) - 1) * 100
        speedups[pf] = vals
    return speedups


cp_data = get_pf_speedups("cp")
wp_data = get_pf_speedups("wp")

# -- Build x-axis: prefetcher names repeated per suite --
all_names = []
all_cp_vals = []
all_wp_vals = []
suite_boundaries = []
suite_labels = []
x_pos = 0

for suite in ph.SUITES:
    start = x_pos
    for pf, label in PREFETCHERS.items():
        all_names.append(label)
        all_cp_vals.append(cp_data[pf][suite])
        all_wp_vals.append(wp_data[pf][suite])
        x_pos += 1
    suite_labels.append((start + len(PREFETCHERS) / 2, ph.SUITE_DISPLAY[suite]))
    suite_boundaries.append(x_pos - 0.5)

Y_MIN = -15.1
Y_MAX = 6.2
Y_DTICK = 5

# -- Plot --
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_cp_vals,
        name="No-WP",
        marker_color=CP_BAR_COLOR,
    )
)

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_wp_vals,
        name="ED-WP",
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
    )
)

# Suite dividers and labels
for boundary in suite_boundaries[:-1]:
    fig.add_vline(x=boundary, line_width=2, line_dash="dash", line_color="gray")
for xpos, label in suite_labels:
    fig.add_annotation(
        x=xpos,
        y=1,
        text=label,
        showarrow=False,
        xref="x",
        yref="paper",
        font=dict(size=FONT_SIZE, color="black"),
    )

# Overflow annotations
_ann_ax = [-30, 30, 100]
_ann_xshift = [-15, 15, 10]
_ann_xanchor = ["right", "left", "right"]
_ann_idx = 0
for values in [all_cp_vals, all_wp_vals]:
    for i, val in enumerate(values):
        if val > Y_MAX or val < Y_MIN:
            fig.add_annotation(
                x=i,
                xshift=_ann_xshift[_ann_idx] if _ann_idx < len(_ann_xshift) else 0,
                y=-12,
                text=f"{val:.1f}",
                showarrow=True,
                arrowhead=2,
                arrowwidth=2,
                ax=_ann_ax[_ann_idx] if _ann_idx < len(_ann_ax) else 0,
                ay=-30 if val > 0 else 30,
                xanchor=(
                    _ann_xanchor[_ann_idx] if _ann_idx < len(_ann_xanchor) else "center"
                ),
                font=dict(size=FONT_SIZE * 0.85),
                textangle=0,
            )
            _ann_idx += 1

# fig.add_annotation(
#     x=0.99,
#     y=1.05,
#     text="(c)",
#     showarrow=False,
#     xref="paper",
#     yref="paper",
#     font=dict(size=FONT_SIZE, color="black", family="Arial, sans-serif"),
#     xanchor="right",
#     yanchor="top",
# )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(
            tickvals=list(range(len(all_names))),
            ticktext=all_names,
            range=[-0.5, len(all_names) - 0.5],
        ),
        yaxis=dict(title="Speedup (%)", range=[Y_MIN, Y_MAX], dtick=Y_DTICK, tick0=0),
        showlegend=False,
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### LLC Repl Policies

In [29]:
import math

importlib.reload(ph)

# -- Config --
FIG_NAME = "llc_repl_plot"
FONT_SIZE = ph.ISPASS_FONT_SIZE

REPL_POLICIES = {
    "llc-ship": "SHIP",
    "llc-srrip": "SRRIP",
    "llc-drrip": "DRRIP",
    "llc-mockingjay": "Mockingjay",
}

CP_BAR_COLOR = "mediumseagreen"
WP_BAR_COLOR = "darkorange"


# -- Compute speedups --
def get_repl_speedups(version):
    speedups = {}
    for rp, label in REPL_POLICIES.items():
        vals = {}
        for suite in ph.SUITES:
            baseline = results[f"{suite}_{version}_default_df"]["IPC"]["gmean"]
            rp_ipc = results[f"{suite}_{version}_{rp}_df"]["IPC"]["gmean"]
            vals[suite] = ((rp_ipc / baseline) - 1) * 100
        speedups[rp] = vals
    return speedups


cp_data = get_repl_speedups("cp")
wp_data = get_repl_speedups("wp")

# -- Build x-axis: policy names repeated per suite --
all_names = []
all_cp_vals = []
all_wp_vals = []
suite_boundaries = []
suite_labels = []
x_pos = 0

for suite in ph.SUITES:
    start = x_pos
    for rp, label in REPL_POLICIES.items():
        all_names.append(label)
        all_cp_vals.append(cp_data[rp][suite])
        all_wp_vals.append(wp_data[rp][suite])
        x_pos += 1
    suite_labels.append((start + len(REPL_POLICIES) / 2, ph.SUITE_DISPLAY[suite]))
    suite_boundaries.append(x_pos - 0.5)

all_vals = all_cp_vals + all_wp_vals
Y_MIN = 0
Y_MAX = 12
Y_DTICK = 4

# -- Plot --
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_cp_vals,
        name="No-WP",
        marker_color=CP_BAR_COLOR,
    )
)

fig.add_trace(
    go.Bar(
        x=list(range(len(all_names))),
        y=all_wp_vals,
        name="ED-WP",
        marker=dict(
            color="rgba(0,0,0,0)",
            line=dict(color=WP_BAR_COLOR, width=2),
            pattern=dict(shape="/", fgcolor=WP_BAR_COLOR, solidity=0.4),
        ),
    )
)

# Suite dividers and labels
for boundary in suite_boundaries[:-1]:
    fig.add_vline(x=boundary, line_width=4, line_dash="dash", line_color="gray")
for xpos, label in suite_labels:
    fig.add_annotation(
        x=xpos,
        y=0.95,
        text=label,
        showarrow=False,
        xref="x",
        yref="paper",
        font=dict(size=FONT_SIZE, color="black"),
    )

# (d) label
# fig.add_annotation(
#     x=0.99,
#     y=1.05,
#     text="(d)",
#     showarrow=False,
#     xref="paper",
#     yref="paper",
#     font=dict(size=FONT_SIZE, color="black", family="Arial, sans-serif"),
#     xanchor="right",
#     yanchor="top",
# )

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(
            tickvals=list(range(len(all_names))),
            ticktext=all_names,
            range=[-0.5, len(all_names) - 0.5],
        ),
        yaxis=dict(title="Speedup (%)", range=[Y_MIN, Y_MAX], dtick=Y_DTICK, tick0=0),
        showlegend=False,
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT, fmt="svg")

### Simulation Overhead

In [ ]:
importlib.reload(ph)

# -- Config --
FIG_NAME = "simulation_overhead"
FONT_SIZE = ph.ISPASS_FONT_SIZE
SUITE_LABELS = [ph.SUITE_DISPLAY[s] for s in ph.SUITES]

# -- gem5 simulation times (sum of simpoints per benchmark, then averaged) --
gem5_times = {
    "lcf": {
        "web-search": 1066.56,
        "media-stream": 1832.95,
        "specjbb": 2809.91,
        "wikipedia": 1809.64,
        "finagle-http": 1861.85,
        "speedometer2.0": 1553.5,
        "data-serving": 2144.28,
        "kafka": 1489.56,
        "tpcc": 1745.23,
        "cassandra": 1590.09,
        "finagle-chirper": 2100.05,
        "verilator-bolted": 1582.68,
        "tomcat": 2162.9,
    },
    "gaps": {
        "sssp": 2430.43,
        "pr": 4143.26,
        "prspmv": 4761.63,
        "cc": 4288.41,
        "bc": 4386.38,
        "ccsv": 5093.96,
        "bfs": 3731.67,
        "tc": 2923.82,
    },
    "spec": {
        "505.mcf": 16450.39,
        "525.x264": 4698.28,
        "531.deepsjeng": 9400.89,
        "541.leela": 8859.02,
        "548.exchange2": 4947.04,
        "557.xz": 6290.70,
    },
}

# -- Compute overheads --
wp_overheads = []
wpa_overheads = []
gem5_overheads = []
for suite in ph.SUITES:
    cp_time = results[f"{suite}_cp_default_df"]["Simulation_Time"]["amean"]
    wp_time = results[f"{suite}_wp_default_df"]["Simulation_Time"]["amean"]
    wpa_time = results[f"{suite}_wpa_default_df"]["Simulation_Time"]["amean"]
    gem5_avg = sum(gem5_times[suite].values()) / len(gem5_times[suite])
    wp_overheads.append(((wp_time - cp_time) / cp_time) * 100)
    wpa_overheads.append(((wpa_time - cp_time) / cp_time) * 100)
    gem5_overheads.append(((gem5_avg - cp_time) / cp_time) * 100)

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=SUITE_LABELS,
        y=wp_overheads,
        name="ED-WP",
        marker_color="darkorange",
    )
)
# fig.add_trace(
#     go.Bar(
#         x=SUITE_LABELS,
#         y=wpa_overheads,
#         name="ChampSim WPA",
#         marker_color="steelblue",
#     )
# )
fig.add_trace(
    go.Bar(
        x=SUITE_LABELS,
        y=gem5_overheads,
        name="gem5",
        marker=dict(
            color="indianred",
            line=dict(color="indianred", width=1),
            pattern=dict(
                shape="|",
                fgcolor="indianred",
                bgcolor="white",
                fillmode="overlay",
                solidity=0.3,
            ),
        ),
    )
)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickangle=-25),
        yaxis=dict(title="Simulation Overhead (%)", range=[0, 71]),
        legend=dict(
            orientation="v",
            x=0.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1,
        ),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT)

### IPC gem5 v ChampSim No-WP v ChampSim ED-WP

In [ ]:
importlib.reload(ph)

# -- Config --
FIG_NAME = "gem5_ipc_comparison"
FONT_SIZE = ph.ISPASS_FONT_SIZE

# -- gem5 IPC (100M instructions / cycles) --
gem5_ipc_data = {
    # LCF
    "web-search": 100_000_000 / 16473707,
    "media-stream": 100_000_000 / 95746267,
    "specjbb": 100_000_000 / 266164899,
    "wikipedia": 100_000_000 / 77447466,
    "finagle-http": 100_000_000 / 81383219,
    "speedometer2.0": 100_000_000 / 47488791,
    "data-serving": 100_000_000 / 103769915,
    "kafka": 100_000_000 / 79159920,
    "tpcc": 100_000_000 / 61439096,
    "cassandra": 100_000_000 / 97091994,
    "finagle-chirper": 100_000_000 / 104312048,
    "verilator-bolted": 100_000_000 / 81672355,
    "tomcat": 100_000_000 / 116482431,
    # GAP
    "sssp": 100_000_000 / 244354558,
    "pr": 100_000_000 / 476048849,
    "prspmv": 100_000_000 / 485240310,
    "cc": 100_000_000 / 780467232,
    "bc": 100_000_000 / 797901088,
    "ccsv": 100_000_000 / 319483518,
    "bfs": 100_000_000 / 287132347,
    "tc": 100_000_000 / 155976750,
    # SPEC (IPC = 100M / weighted_cycles, weighted by simpoint weights)
    "505.mcf": 0.619264,
    "525.x264": 4.714522,
    "531.deepsjeng": 3.310158,
    "541.leela": 2.979212,
    "548.exchange2": 3.772270,
    "557.xz": 1.738605,
}


# -- Metric --
def ipc_value(version):
    def fn(results, suite, benchmarks):
        ipc = results[f"{suite}_{version}_default_df"]["IPC"]
        return [ipc.get(bm, 0) for bm in benchmarks]

    return fn


def gem5_ipc_fn(results, suite, benchmarks):
    return [gem5_ipc_data.get(bm, 0) for bm in benchmarks]


# -- Data --
all_benchmarks, cp_values, suite_boundaries, suite_labels = ph.get_metric_across_suites(
    results,
    ipc_value("cp"),
    include_amean=False,
)
_, wp_values, _, _ = ph.get_metric_across_suites(
    results,
    ipc_value("wp"),
    include_amean=False,
)
_, wpa_values, _, _ = ph.get_metric_across_suites(
    results,
    ipc_value("wpa"),
    include_amean=False,
)
_, gem5_values, _, _ = ph.get_metric_across_suites(
    results,
    gem5_ipc_fn,
    include_amean=False,
)
display_names = ph.get_display_names(all_benchmarks)
tick_labels = ph.get_tick_labels(all_benchmarks)

# -- Plot --
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=display_names,
        y=gem5_values,
        name="gem5",
        marker_color="indianred",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=cp_values,
        name="No-WP",
        marker_color="mediumseagreen",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wp_values,
        name="WP",
        marker_color="darkorange",
    )
)
fig.add_trace(
    go.Bar(
        x=display_names,
        y=wpa_values,
        name="WPA",
        marker_color="steelblue",
    )
)

ph.add_suite_dividers(fig, suite_boundaries, suite_labels)

fig.update_layout(
    **ph.base_layout(
        font_size=FONT_SIZE,
        xaxis=dict(tickvals=display_names, ticktext=tick_labels),
        yaxis=dict(title="IPC"),
    )
)

fig.show()
ph.save_fig(fig, FIG_NAME, ph.ISPASS_ONE_COL_WIDTH, ph.ISPASS_ONE_COL_HEIGHT)